# 09 — External Transfer: Project External Dataset onto ABCD PC1

Projects an external (replication) dataset's longitudinal delta-Z scores onto the PC1 loadings derived from the ABCD healthy normative sample (output of `08_pca_brain_scores.ipynb`).

**Normative transfer approach:** The ABCD PC1 axis defines the normative brain-change space. External subjects are positioned in that space via a weighted dot product — no re-fitting on the external data.

**Input:**
- `ABCD_LOADINGS_DIR` — PC1 loadings from `08_pca_brain_scores.ipynb`
- `EXTERNAL_DELTA_CSV` — external dataset longitudinal delta-Z file (`ID` index + ROI columns)

**Output:** `<dataset>_PC1_projected.csv` — one projected PC1 score per external subject

In [ ]:
# ── CONFIG ──────────────────────────────────────────────────────────────────
# Directory containing ABCD PC1 loadings (output of 08_pca_brain_scores.ipynb)
ABCD_LOADINGS_DIR  = 'outputs/pca/longitudinal'

# Which longitudinal delta-Z pair's loadings to use for the transfer
LOADINGS_PAIR      = 'V1_V4'   # e.g. 'V1_V2', 'V1_V4'

# External dataset: path to delta-Z CSV  (must have 'ID' index + ROI columns)
EXTERNAL_DELTA_CSV = 'data/external/EXTERNAL_DeltaZ.csv'

# Output directory
OUTPUT_DIR         = 'outputs/pca/external_transfer'

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

os.makedirs(OUTPUT_DIR, exist_ok=True)

## Step 1 — Load ABCD PC1 loadings

In [ ]:
loadings_path = os.path.join(ABCD_LOADINGS_DIR, f'PC1_Loadings_{LOADINGS_PAIR}_healthy.csv')
abcd_loadings = pd.read_csv(loadings_path, index_col=0)
w_abcd        = abcd_loadings['PC1_Loading']

print(f'ABCD loadings ({LOADINGS_PAIR}): {len(w_abcd)} ROIs')
print(f'Source: {loadings_path}')

## Step 2 — Load external dataset delta-Z

In [ ]:
df_ext = pd.read_csv(EXTERNAL_DELTA_CSV)
if 'ID' in df_ext.columns:
    df_ext = df_ext.set_index('ID')

print(f'External data: {len(df_ext)} subjects | {df_ext.shape[1]} columns')

## Step 3 — Align ROIs and project

Only ROIs present in both the ABCD loadings and the external dataset are used.  
External data is standardised before projection (z-score within the external sample).

In [ ]:
common_rois = df_ext.columns.intersection(w_abcd.index).tolist()
print(f'Common ROIs: {len(common_rois)} / {len(w_abcd)} ABCD ROIs')

if len(common_rois) == 0:
    raise RuntimeError(
        'No overlapping ROIs — check that external delta-Z column names match ABCD ROI names.'
    )

Z_ext  = df_ext[common_rois].dropna()
w_sub  = w_abcd.loc[common_rois].values

scaler     = StandardScaler()
Z_scaled   = scaler.fit_transform(Z_ext)
pc1_scores = Z_scaled @ w_sub

print(f'Projected {len(pc1_scores)} subjects onto ABCD PC1 ({LOADINGS_PAIR})')

## Step 4 — Save projected scores

In [ ]:
out_name  = os.path.splitext(os.path.basename(EXTERNAL_DELTA_CSV))[0]
df_scores = pd.DataFrame(
    {'PC1_ABCD_projected': pc1_scores},
    index=Z_ext.index,
)
df_scores.index.name = 'ID'

out_path = os.path.join(OUTPUT_DIR, f'{out_name}_PC1_projected.csv')
df_scores.to_csv(out_path)

print(f'Saved: {out_path}  ({len(df_scores)} subjects)')
print(f'ROIs used: {len(common_rois)} / {len(w_abcd)} (ABCD) — {len(w_abcd) - len(common_rois)} dropped (not in external data)')